# 28 — Concept scaling: steering with 3 concurrent concepts

How does multi-concept steering behave as the number of concurrently steered
concepts grows? This notebook measures the n=2 → n=3 step on the validated
concept pool of notebook 27 (`resources/27_concept_pool.json`); notebook 29
will extend the same design to n=4. Every guide here is individually a
proven steerer, so joint failures reflect *composition*, not weak
constituents — the confound the old notebook-15 pairs could not exclude.

**Nested design, selected offline from the pool file** (the pool carries the
full pairwise ρ, so group selection is deterministic JSON arithmetic):

- **10 base pairs** evenly spaced along the pool's pairwise-ρ spectrum, with
  each concept used at most twice (diversity);
- each pair is extended with a **third concept at three entanglement
  strata** — lowest / median / highest `max(ρ(c,a), ρ(c,b))` over the rest
  of the pool — giving 30 nested triples (`add_regime` ∈ distant/mid/close);
- **both arms run fresh here**: the same pairs at n=2 and their triples at
  n=3, same model, prompts, k, steps — so the 2→3 change is within-pair.

**Held fixed** (notebook-24 showcase settings): Llama-3.1-8B-Instruct AWQ,
k=4, steps=24, the 10 showcase prompts, frame blending (simple blend) /
score voting (z-scored weighted vote) / concept scheduling (round-robin).
The fixed 24-step budget is intentional: under scheduling each concept is
active ~8 steps at n=3 vs 12 at n=2 — capacity contention is part of what is
being measured.

**Interpretation caveats.** `joint = min` over n deltas is stochastically
smaller at larger n even under identical marginals — compare method
*rankings* within each n and each method's own 2→3 drop, not raw joint
values across n. The `add_regime` breakdown separates "adding an unrelated
third concept" from "adding one entangled with the pair".


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept
from frames.representations.schedulers import RoundRobin

In [ ]:
MIN_LEMMAS = 11  # Llama concepts draw on 8 WordNet languages (multiconcept standard)
MAX_TOKENS = 3
K = 4
STEPS = 24
N_BASE_PAIRS = 10
MAX_USES_PER_CONCEPT = 2

MODEL_ID = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4"
MODEL_KEY = "llama-3.1-8b"

POOL = json.loads(Path("resources/27_concept_pool.json").read_text())
POOL_NAMES = [r["synset"] for r in POOL["pool"]]
DELTA1 = {r["synset"]: r["delta"] for r in POOL["pool"]}  # n=1 reference


def rho(a: str, b: str) -> float:
    key = f"{a}|{b}" if f"{a}|{b}" in POOL["pairwise_rho"] else f"{b}|{a}"
    return POOL["pairwise_rho"][key]


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

LOG_PATH = Path("resources/28_scaling3.jsonl")
GROUPS_PATH = Path("resources/28_scaling_groups.json")
LOG_PATH.unlink(missing_ok=True)
print(f"pool: {len(POOL_NAMES)} validated concepts")

In [ ]:
# 10 base pairs evenly spaced along the pool's rho spectrum, each concept
# used at most twice
all_pairs = sorted(
    (
        {"a": a, "b": b, "pair_rho": rho(a, b)}
        for i, a in enumerate(POOL_NAMES)
        for b in POOL_NAMES[i + 1 :]
    ),
    key=lambda p: p["pair_rho"],
)
targets = [
    all_pairs[round(i * (len(all_pairs) - 1) / (N_BASE_PAIRS - 1))]["pair_rho"]
    for i in range(N_BASE_PAIRS)
]
PAIRS, uses = [], {}
for t in targets:
    for p in sorted(all_pairs, key=lambda p: abs(p["pair_rho"] - t)):
        key = (p["a"], p["b"])
        if key in {(q["a"], q["b"]) for q in PAIRS}:
            continue
        if uses.get(p["a"], 0) >= MAX_USES_PER_CONCEPT:
            continue
        if uses.get(p["b"], 0) >= MAX_USES_PER_CONCEPT:
            continue
        PAIRS.append(p)
        uses[p["a"]] = uses.get(p["a"], 0) + 1
        uses[p["b"]] = uses.get(p["b"], 0) + 1
        break
if len(PAIRS) < N_BASE_PAIRS:
    raise RuntimeError("usage cap prevented an even pair spread")

# extend each pair with a third concept at three entanglement strata
TRIPLES = []
for p in PAIRS:
    a, b = p["a"], p["b"]
    ranked = sorted(
        (c for c in POOL_NAMES if c not in (a, b)),
        key=lambda c: max(rho(c, a), rho(c, b)),
    )
    picks = {"distant": ranked[0], "mid": ranked[len(ranked) // 2], "close": ranked[-1]}
    for add_regime, c in picks.items():
        TRIPLES.append(
            {
                "a": a,
                "b": b,
                "c": c,
                "add_regime": add_regime,
                "add_rho": max(rho(c, a), rho(c, b)),
                "mean_rho": (rho(a, b) + rho(a, c) + rho(b, c)) / 3,
                "pair_rho": p["pair_rho"],
            }
        )

GROUPS_PATH.write_text(
    json.dumps(
        {
            "config": {
                "pool": str(Path("resources/27_concept_pool.json")),
                "n_base_pairs": N_BASE_PAIRS,
                "max_uses_per_concept": MAX_USES_PER_CONCEPT,
                "strategy": "pairs evenly spaced in pool rho (usage-capped), "
                "each extended with a third at distant/mid/close "
                "entanglement (max rho to the pair)",
            },
            "pairs": PAIRS,
            "triples": TRIPLES,
        },
        indent=1,
    )
)
print(
    f"{len(PAIRS)} pairs (rho {PAIRS[0]['pair_rho']:.2f}..{PAIRS[-1]['pair_rho']:.2f}), {len(TRIPLES)} triples -> {GROUPS_PATH}"
)

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(device_map="cuda:0", id=MODEL_ID)
prompts = [chat(u, a) for u, a in QUESTIONS]

harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)

used = sorted({s for g in PAIRS + TRIPLES for s in (g["a"], g["b"], g.get("c")) if s})
concept = {name: fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS) for name in used}
print(f"{len(concept)} concepts built")


def run_group(group: dict) -> None:
    synsets = [s for s in (group["a"], group["b"], group.get("c")) if s]
    cs = [concept[s] for s in synsets]
    methods = {
        "blend": lambda: fur.generate_with_topk_guide(
            prompts, guide=Concept.average(cs), k=K, steps=STEPS
        ),
        "vote": lambda: fur.generate_with_topk_multi_guide(
            prompts, guides=cs, k=K, steps=STEPS, normalize="zscore"
        ),
        "schedule": lambda: fur.generate_with_topk_multi_guide(
            prompts, guides=cs, k=K, steps=STEPS, scorer=RoundRobin()
        ),
    }
    for method, generate in methods.items():
        torch.cuda.empty_cache()
        texts, probe = generate()
        harness.evaluate(
            prompts,
            texts,
            synsets,
            config={
                "model": MODEL_KEY,
                "method": method,
                "n": len(synsets),
                **group,
            },
            probe=probe,
            baseline_texts=baseline_texts,
            concepts=dict(zip(synsets, cs)),
            angles=True,
        )
    label = " + ".join(s.split(".")[0] for s in synsets)
    print(f"n={len(synsets)} {label} ({group.get('add_regime', 'pair')}): done")


for group in [*PAIRS, *TRIPLES]:
    run_group(group)

In [ ]:
parsed = [json.loads(line) for line in LOG_PATH.read_text().strip().split("\n")]
assert {r["config"]["n"] for r in parsed} == {2, 3}

rows = []
for r in parsed:
    c = r["config"]
    synsets = [s for s in (c["a"], c["b"], c.get("c")) if s]
    deltas = [r["expression"][s]["delta"] for s in synsets]
    cosines = [r["angles"][s]["delta"]["token_cos"] for s in synsets]
    rows.append(
        {
            "method": c["method"],
            "n": c["n"],
            "joint": min(deltas),
            "all_positive": min(deltas) > 0,
            # metric v3: bounded/scale-free joint — comparable across n
            # (up to min-shrinkage) and across models
            "joint_cos": min(cosines),
            "all_positive_cos": min(cosines) > 0,
            "mean_single_delta": sum(DELTA1[s] for s in synsets) / len(synsets),
            "add_regime": c.get("add_regime"),
            "mean_rho": c.get("mean_rho", c["pair_rho"]),
            "ppl_ratio": r["ppl_ratio"],
        }
    )
results = pd.DataFrame(rows)
print(f"gate: {results.groupby('n').size().to_dict()} records")

summary = (
    results.groupby(["method", "n"])
    .agg(
        joint=("joint", "mean"),
        all_positive=("all_positive", "mean"),
        joint_cos=("joint_cos", "mean"),
        all_positive_cos=("all_positive_cos", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        n_runs=("joint", "size"),
    )
    .reset_index()
)
summary.to_csv("resources/28_scaling3_summary.csv", index=False)

by_regime = (
    results[results["n"] == 3]
    .groupby(["method", "add_regime"])
    .agg(
        joint=("joint", "mean"),
        all_positive=("all_positive", "mean"),
        joint_cos=("joint_cos", "mean"),
    )
    .reset_index()
)
summary.round(4)

In [ ]:
order = ["blend", "vote", "schedule"]
colors = {"blend": "tab:blue", "vote": "tab:orange", "schedule": "tab:green"}
ns = [2, 3]
regimes = ["distant", "mid", "close"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, (metric, label) in zip(
    axes[:2],
    (
        ("joint", "mean joint expression gain"),
        ("joint_cos", "mean joint token-cosine gain (scale-free)"),
    ),
):
    for j, method in enumerate(order):
        vals = [
            summary[(summary["method"] == method) & (summary["n"] == n)][metric].iloc[0]
            for n in ns
        ]
        ax.bar(
            [i + (j - 1) * 0.25 for i in range(len(ns))],
            vals,
            width=0.24,
            color=colors[method],
            label=method,
        )
    ax.set_xticks(range(len(ns)), [f"n={n}" for n in ns])
    ax.set_ylabel(label)
    ax.axhline(0, color="gray", lw=0.8)
axes[0].legend(fontsize=8)

for j, method in enumerate(order):
    vals = [
        by_regime[(by_regime["method"] == method) & (by_regime["add_regime"] == reg)][
            "joint"
        ].iloc[0]
        for reg in regimes
    ]
    axes[2].bar(
        [i + (j - 1) * 0.25 for i in range(len(regimes))],
        vals,
        width=0.24,
        color=colors[method],
    )
axes[2].set_xticks(range(len(regimes)), regimes)
axes[2].set_ylabel("mean joint gain at n=3, by added-concept entanglement")
axes[2].axhline(0, color="gray", lw=0.8)

plt.suptitle("Steering as concurrent concepts scale (validated pool, nested)")
plt.tight_layout()
plt.savefig("resources/28_scaling3.png", dpi=150)
plt.show()

print("method ranking by joint (expression / token-cosine):")
for n in ns:
    by_expr = summary[summary["n"] == n].sort_values("joint", ascending=False)
    by_cos = summary[summary["n"] == n].sort_values("joint_cos", ascending=False)
    print(
        f"  n={n}: "
        + " > ".join(by_expr["method"])
        + "   |   "
        + " > ".join(by_cos["method"])
    )

print("\nall-improved rates (expression / token-cosine):")
for _, row in summary.iterrows():
    print(
        f"  n={row['n']} {row['method']:9s} "
        f"{row['all_positive']:.0%} / {row['all_positive_cos']:.0%}"
    )

In [ ]:
def answer(text: str) -> str:
    marker = "assistant<|end_header_id|>"
    return text.split(marker)[-1].strip() if marker in text else text


for regime in ("distant", "close"):
    for r in parsed:
        c = r["config"]
        if (
            c["method"] == "blend"
            and c.get("add_regime") == regime
            and "short story" in r["prompt"]
        ):
            deltas = {
                k.split(".")[0]: round(v["delta"], 2)
                for k, v in r["expression"].items()
            }
            print(f"[blend / {regime} third] {deltas}")
            print("   ", answer(r["text"])[:200], "\n")
            break